In [ ]:
!pip install transformers sentencepiece


In [ ]:
from transformers import AutoTokenizer, AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

prompt = "Explain Artificial Intelligence in simple words."

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

artificial intelligence is a type of artificial intelligence.


Role-Based Prompting

In [ ]:
prompt2 = """
You are a senior data scientist.
Explain overfitting using a real-world example.
"""

inputs2 = tokenizer(prompt2, return_tensors="pt")
outputs2 = model.generate(**inputs2, max_new_tokens=120)
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

Overfitting is a problem in the field of statistics.


Few-Short Prompting

In [ ]:
# ====== PROMPT 2 (Passive → Active voice) ======
prompt2 = """
Convert the following sentences to active voice.

Passive: The cake was eaten by John.
Passive: The ball was thrown by Riya.
"""

inputs2 = tokenizer(prompt2, return_tensors="pt")
outputs2 = model.generate(**inputs2, max_new_tokens=100)

print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

The ball was thrown by Riya.


Format-Controlled Prompting

In [ ]:
prompt = """
Explain Machine Learning in a table.

Columns:
Concept | Meaning | Example
"""

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(inputs["input_ids"], max_new_tokens=200)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Table of Contents: Machine Learning Table of Contents:


🔴 5. Chain-of-Thought (CoT) Prompting


In [ ]:
!pip install transformers sentencepiece --quiet

In [ ]:
# -------------------------
# FLAN-T5: step-by-step solver
# -------------------------
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
prompt = """
Solve step-by-step:
A train travels 120 km in 2 hours.
What is the speed?
"""
# Tokenize and move inputs to device
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}
# Generate — use beam search for clearer step-by-step answer
outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=4,
    early_stopping=True
)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model output:\n")
print(answer)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model output:

S = 120 * 2 = 480 kmph. S = 480 / 120 = 5 kmph. The answer: 5 .


model to “think out loud” for better reasoning.

💡 Conclusion: Prompt Engineering = Instruction + Role + Context + Examples + Format + Reasoning The better your prompt → the better the model’s output